In [0]:
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType


def get_spark():
    return (
        SparkSession.builder
        .master("local[1]")
        .appName("test_maritime_ingestion")
        .getOrCreate()
    )


def test_combined_sources_latest_metric_extraction():
    spark = get_spark()
    payload = [
        {
            "selfReportedInfo": json.dumps([{"id": "v-100", "ssvid": "123456789"}]),
            "registryInfo": json.dumps([]),
            "combinedSourcesInfo": json.dumps([
                {
                    "vesselId": "v-100",
                    "bestLengthM": [
                        {"value": 28.5, "yearTo": 2022},
                        {"value": 36.18, "yearTo": 2024},
                    ],
                }
            ]),
        }
    ]
    df = spark.createDataFrame(payload)

    parsed = df.withColumn(
        "metrics",
        F.from_json(
            F.col("combinedSourcesInfo"),
            "array<struct<vesselId:string,bestLengthM:array<struct<value:double,yearTo:int>>>>",
        ),
    ).withColumn(
        "latest_length",
        F.expr("array_sort(metrics[0].bestLengthM, (l, r) -> case when l.yearTo > r.yearTo then -1 else 1 end)[0].value"),
    )

    row = parsed.select("latest_length").first()
    assert row["latest_length"] == 36.18


def test_registry_only_vessel_fallback():
    spark = get_spark()
    payload = [
        {
            "selfReportedInfo": json.dumps([]),
            "registryInfo": json.dumps([{"id": "reg-999", "shipname": "OCEAN STAR", "flag": "NLD"}]),
        }
    ]
    df = spark.createDataFrame(payload)

    parsed = (
        df.withColumn("sr", F.from_json("selfReportedInfo", "array<struct<id:string,shipname:string>>"))
        .withColumn("rg", F.from_json("registryInfo", "array<struct<id:string,shipname:string,flag:string>>"))
        .withColumn("vessel_id", F.coalesce(F.get(F.col("sr"), 0)["id"], F.get(F.col("rg"), 0)["id"]))
        .withColumn("shipname", F.coalesce(F.get(F.col("sr"), 0)["shipname"], F.get(F.col("rg"), 0)["shipname"]))
    )

    row = parsed.select("vessel_id", "shipname").first()
    assert row["vessel_id"] == "reg-999"
    assert row["shipname"] == "OCEAN STAR"


def test_all_string_bronze_schema_enforced():
    schema = StructType([
        StructField("selfReportedInfo", StringType(), True),
        StructField("registryInfo", StringType(), True),
        StructField("combinedSourcesInfo", StringType(), True),
        StructField("_rescued_data", StringType(), True),
    ])
    assert all(isinstance(field.dataType, StringType) for field in schema.fields)